# Modül 2 — Model Değerlendirme

**Eğitmen: Elif Özkan**

[LinkedIn](https://www.linkedin.com/in/eliffozkan/) · [GitHub](https://github.com/eliffozkan) · [Eğitim reposu](https://github.com/eliffozkan/GDGC-AI_ML_Course-26_27)

[YouTube — Fütürist AI (kaynak)](https://www.youtube.com/@F%C3%BCt%C3%BCrist_AIntelligence)

[Google Colab’da aç](https://colab.research.google.com/github/eliffozkan/GDGC-AI_ML_Course-26_27/blob/main/Module_2_Metrics/Module_2_Metrics_Code.ipynb)

**Ders 1 (75 dk):** confusion matrix/accuracy 15, precision/recall/F1 30, ROC grafik yorumu 10, train/val/test 20 dk.

**Ders 2 (75 dk):** K-Fold/Stratified 20, genelleme ve kısa belirsizlik yorumu 15, Grid/Random Search 30, ödev bağlantısı 10 dk.

**Çalıştırma:** Temiz oturumda hücreleri üstten alta çalıştırın. Kod çıktılarını ilgili açıklamayla birlikte yorumlayın.

## Ders 1 — Tahminleri ölçmek

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score
np.random.seed(42)

In [ ]:
y = np.array([0, 0, 0, 1, 1, 1, 1, 0])
y_predicted = np.array([0, 1, 0, 1, 0, 1, 1, 0])
print("Accuracy:", accuracy_score(y, y_predicted))
print("Confusion matrix (gerçek satır, tahmin sütun):\n", confusion_matrix(y, y_predicted))
ConfusionMatrixDisplay.from_predictions(y, y_predicted); plt.show()

# Precision

In [ ]:
from sklearn.metrics import precision_score
precision_score(y,y_predicted)

In [ ]:
def precision(y:np.array,y_pred:np.array):
    tp = np.sum((y == 1) & (y_pred == 1))
    fp = np.sum((y == 0) & (y_pred==1))
    if tp + fp == 0: return 0.0
    prec = tp / (tp+fp)
    return prec 

In [ ]:
precision(y,y_predicted)

Precision: tahmin edilen pozitiflerin ne kadarı doğru? `tp+fp=0` ise örneğimiz 0 döndürür. Recall ise gerçek pozitiflerin kaçının yakalandığını ölçer.

# Recall

In [ ]:
from sklearn.metrics import recall_score
recall_score(y,y_predicted)

In [ ]:
def recall(y:np.array,y_pred:np.array):
    tp = np.sum((y == 1) & (y_pred == 1))
    fn = np.sum((y == 1) & (y_pred==0))
    if tp + fn == 0: return 0.0
    rec = tp / (tp+fn)
    return rec 

In [ ]:
recall(y,y_predicted)

# F1 — binary ve multiclass
F1, precision ve recall’un harmonik ortalamasıdır. Binary örnekte pozitif sınıf 1’dir. Macro sınıfları eşit, weighted sınıf destekleri oranında ağırlıklandırır; micro toplam TP/FP/FN üzerinden hesaplar.

In [ ]:
print("Binary F1:", f1_score(y, y_predicted))
y = np.array([0, 0, 1, 1, 2, 2])
y_predicted = np.array([0, 1, 1, 0, 2, 2])

In [ ]:
f1_score(y,y_predicted,average="macro")

In [ ]:
print("Micro:", f1_score(y, y_predicted, average="micro"))
print("Weighted:", f1_score(y, y_predicted, average="weighted"))

# ROC-AUC

In [ ]:
from sklearn.metrics import roc_curve, auc , roc_auc_score

In [ ]:
y = np.array([0, 0, 1, 1, 0, 1])
y_predicted = np.array([0.1, 0.4, 0.35, 0.8, 0.2, 0.7])  # Temsili olasılık skorları
fpr, tpr, thresholds = roc_curve(y, y_predicted)

In [ ]:
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
auc_score = roc_auc_score(y, y_predicted)
print("ROC-AUC:", auc_score)

ROC’da y ekseni recall/TPR, x ekseni FPR’dir. Sol üst daha iyidir; diyagonal rastgele sıralamayı gösterir. AUC sınıf etiketlerinden değil, olasılık/karar skorlarından yorumlanır.

# Train-Val-Test

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
X, y = load_breast_cancer(return_X_y=True)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=42)

In [ ]:
print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
model = LogisticRegression(max_iter=1000, solver="liblinear", random_state=42).fit(X_train, y_train)
y_predicted = model.predict(X_val)
print("Validation accuracy:", accuracy_score(y_val, y_predicted))
print("Validation F1 (1=iyi huylu):", f1_score(y_val, y_predicted))

Train modeli eğitir, validation seçenekleri karşılaştırır; test nihai değerlendirme için saklanır. `C` düzenlileştirme gücünün tersidir: daha küçük C daha güçlü kısıtlama demektir. Solver burada hazır sayısal çözüm seçeneğidir.

---
## Ders 2 — Güvenilir karşılaştırma ve model seçimi
Önceki dersin LogisticRegression modelini kullanıyoruz; yeni bir model ailesi öğrenmiyoruz.

# K-Fold

In [ ]:
from sklearn.model_selection import KFold
kf = KFold(n_splits=3, shuffle=True, random_state=42)

In [ ]:
for fold, (train_index, val_index) in enumerate(kf.split(X_train_val), 1):
    print(f"Fold {fold}: train={len(train_index)}, validation={len(val_index)}")

# Stratified K-Fold
Sınıf oranlarını fold’larda yaklaşık korur. CV yalnızca eğitim havuzunda yapılır; son test seti ayrıdır.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [ ]:
scores = cross_val_score(LogisticRegression(max_iter=1000, solver="liblinear", random_state=42), X_train_val, y_train_val, cv=skf, scoring="accuracy")
print("Fold accuracy:", scores, "ortalama:", scores.mean(), "std:", scores.std())

Fold skorlarının farklı olması veri bölünmesine duyarlılığı gösterir. Güven aralığı bir tahminin belirsizliğini ifade eder; fold’lar bağımsız olmadığından bu std doğrudan kesin bir güven aralığı değildir. Repeated Stratified K-Fold tekrarlarla değerlendirme yapar; bu dönem uygulamıyoruz.

# Genelleme — train/validation loss
Aşağıdaki değerler gerçek bir eğitim kaydı değil, temsili grafiklerdir. İki kayıp da yüksek kalırsa underfitting; train düşerken validation yükselirse overfitting işareti olabilir.

In [ ]:
train_loss = np.array([1.        , 0.93571429, 0.87142857, 0.80714286, 0.74285714,
       0.67857143, 0.61428571, 0.55      , 0.48571429, 0.42142857,
       0.35714286, 0.29285714, 0.22857143, 0.16428571, 0.1       ,
       0.1       , 0.1       , 0.1       , 0.1       , 0.1       ])
val_loss = np.array([1.05      , 0.98571429, 0.92142857, 0.85714286, 0.79285714,
       0.72857143, 0.66428571, 0.6       , 0.53571429, 0.47142857,
       0.40714286, 0.34285714, 0.27857143, 0.21428571, 0.15      ,
       0.15      , 0.15      , 0.15      , 0.15      , 0.15      ])

In [ ]:
plt.plot(train_loss, label="Train (temsili)")
plt.plot(val_loss, label="Validation (temsili)")
plt.xlabel("Adım"); plt.ylabel("Loss"); plt.legend(); plt.show()

# Grid Search

In [ ]:
from sklearn.model_selection import GridSearchCV
model = LogisticRegression(max_iter=1000, solver="liblinear", random_state=42)
param_grid = {"C": [0.1, 1.0, 10.0]}
grid_search = GridSearchCV(model, param_grid, cv=3, scoring="accuracy")
grid_search.fit(X_train_val, y_train_val)
print("En iyi parametre:", grid_search.best_params_)
print("CV accuracy:", grid_search.best_score_)
for mean_score, params in zip(grid_search.cv_results_["mean_test_score"], grid_search.cv_results_["params"]):
    print(params, round(mean_score, 4))

# Random Search

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
param_dist = {"C": [0.01, 0.1, 1.0, 10.0, 100.0]}
random_search = RandomizedSearchCV(model, param_dist, n_iter=3, cv=3, scoring="accuracy", random_state=42)
random_search.fit(X_train_val, y_train_val)
print("Random Search parametre:", random_search.best_params_)
print("Random Search CV accuracy:", random_search.best_score_)
y_predicted = grid_search.predict(X_test)
print("GridSearch son test accuracy:", accuracy_score(y_test, y_predicted))
print("Son test F1 (1=iyi huylu):", f1_score(y_test, y_predicted))

Grid tüm adayları, Random seçilen aday altkümesini dener. CV skoru model seçimi içindir; test skoru ayrı raporlanır. Bayesian optimization geçmiş sonuçlara göre aday seçen başka bir yaklaşımdır; bu dönem yalnızca ismini biliyoruz. Ödevde GridSearch yalnızca X_train/y_train üzerinde, C=[0.1,1,10], cv=3 ve accuracy ile yapılır.

**Modül materyalleri:** [Module_2_Metrics](https://github.com/eliffozkan/GDGC-AI_ML_Course-26_27/tree/main/Module_2_Metrics)

Ders kodu ve ödev dosyaları bu modül klasöründedir.